### IMPLEMENTATION: **"Optimal Filtering in the ATLAS HADRONIC Tile Calorimeter"**
*ATL-TILECAL-2005-001, 16 February 2005*

In [151]:
import numpy as np
from scipy.optimize import curve_fit, minimize
from scipy.linalg import solve
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# consistent figure style setting
plt.rcParams.update({
      'figure.figsize': (12, 8),
      'font.size': 11,
      'axes.grid': True,
      'grid.alpha': 0.3,
})

#### **Shape Form Function (SFF)**
TileCal signal shape is modeled by the analytical function:

$ \ \ \ \ \ \ \ \ \ \ SF(t) = p + A(\frac{t - \lambda}{\tau})^{\mu} * exp(-\mu \frac{t-\lambda}{\tau}) \ \ \ \ \ \ \ \ \ \         [Eq.35]$


$ p, A, \lambda, \tau, \mu $ are parameter of the fit.

Maximium occurs at:

$  \ \ \ \ \ \ \ \ \ \ t_{max} = \tau + \lambda \ \ \ \ \ \ \ \ \ \ [Eq. 36]$

and the maximum value will be:

$  \ \ \ \ \ \ \ \ \ \ SF(t_{max}) = A \ exp(-\mu) \ \ \ \ \ \ \ \ \ \ [Eq. 37]$

Setting t_max = 0ns and normalize so peak = 1, baseline = 0:

$  \ \ \ \ \ \ \ \ \ \ SF_g(t) = \frac{SF(t)-p}{SF(t_{max})} \ \ \ \ \ \ \ \ \ \ [Eq. 38] $

In [152]:
class ShapeFormFunction:
      """
      Implements the TileCal pulse shape form function. 
      The CIS system sweeps all the time phases in steps of 0.728ns 
      to reconstruct the full analog pulse channel. 
      This shape is then fit by the analytical function in Eq. 35
      """
      def __init__(self, tau=15.0, mu=7.0, pedestal=0.0, gain='high'):
            """
            Parameters reflect typical TileCal pulse shapes.
            High gain pulses are slightly narrower than low gain.
            Physics pulses are ~10% wider than CIS pulses
            Args:
            -----
                  tau: Time constant parameter (in ns)
                  mu: Shape parameter (controls rise and fall asymmetry)
                  pedestal: Baseline offset in ADC counts
                  gain: 'high' or 'low' gain mode
            """
            self.mu = mu
            self.pedestal = pedestal
            self.tau = tau
            self.lam = -tau # lambda = -tau when t_max = 0

            # SF(t_max) = A * exp(-mu)
            # want the raw peak to be some large value (like CIS data aroud ~800 ADC counts)
            self.A_raw = 800.0 / np.exp(-self.mu)

      def sf_raw(self, t):
            """
            Raw Shape Form Function, Eq. (33):
            SF(t) = p + A * ((t - lambda) / tau)^mu * exp(-mu * (t-lambda)/tau)
            Only valid for t >= lambda (signal onset).
            """
            x = (t - self.lam) / self.tau # (t - lambda)/tau
            result = np.zeros_like(np.atleast_1d(t), dtype=float)
            mask = x > 0 # Signal only after onset
            result[mask] = self.pedestal + self.A_raw * (x[mask] ** self.mu) * np.exp(-self.mu * x[mask])
            result[~mask] = self.pedestal
            return result
      
      def sf_normalized(self, t):
            """
            Normalized Shape form Function, Eq. 38: 
                  SF_g(t) = (SF(t) - p) / SF(t_max)
            This gives g(t) with;
                  - Peak value = 1 at t=0 (since we set t_max = 0)
                  - Baseline = 0
            The components of the g vector in the OF equations are sampled from this 
            """
            sf_vals = self.sf_raw(t)
            sf_max = self.sf_raw(np.array([0.0]))[0] # Peak at t_max
            return (sf_vals - self.pedestal)/ (sf_max - self.pedestal)
      
      def sf_derivative(self, t, dt=0.01):
            """
            Numerical derivative of the normalozed shape form: g'(t)
            The g' components are neede for the OF weight calculation.
            They are calculated from the derivative of SF_g(t) at the smae time position as the g components
            """
            return (self.sf_normalized(t+dt) - self.sf_normalized(t-dt)) / (2 * dt)


def plot_shape_form(sff):
    """Visualize the Shape Form Function (reproduces Fig. 2 of the paper)."""
    t = np.linspace(-50, 125, 1000)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Left: Raw SF (like Fig. 2 left - CIS reconstructed shape)
    axes[0].plot(t, sff.sf_raw(t), 'b-', linewidth=2)
    axes[0].set_xlabel('Time (ns)')
    axes[0].set_ylabel('ADC Counts')
    axes[0].set_title('Raw Shape Form Function SF(t)\n[Paper Fig. 2 left]')
    axes[0].axhline(y=sff.pedestal, color='r', linestyle='--', alpha=0.5, label=f'Pedestal = {sff.pedestal}')
    axes[0].legend()
    
    # Middle: Normalized g(t) - Eq. (38)
    g = sff.sf_normalized(t)
    axes[1].plot(t, g, 'b-', linewidth=2)
    axes[1].set_xlabel('Time (ns)')
    axes[1].set_ylabel('Normalized amplitude')
    axes[1].set_title("Normalized SF_g(t) = g(t)\n[Eq. (38)]")
    axes[1].axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
    axes[1].axhline(y=0.0, color='gray', linestyle='--', alpha=0.5)
    
    # Right: Derivative g'(t)
    gp = sff.sf_derivative(t)
    axes[2].plot(t, gp, 'r-', linewidth=2)
    axes[2].set_xlabel('Time (ns)')
    axes[2].set_ylabel("g'(t)")
    axes[2].set_title("Derivative of Shape Form: g'(t)")
    axes[2].axhline(y=0.0, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('fig_shape_form_function.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("[Fig saved] Shape Form Function (Section 3.3)")

#### **Pedestal**

Pedestal is the DC baseline offset of each channel. 
Typical values: 35-65 ADC counts, with RMS -1 count

##### Two approaches;

1. Subtract pedestal from samples before applying OF (used here)
2. Reconstruct pedestal as 3rd parameter (OF2 in Appendix A)

For signal events: pedestal = average of first and last samples

For pedestal events: pedestal = average of all samples

In [153]:
class PedestalCalculator:
      """
            Implements pedestal calculation. 
            The pedestal varies between 35 and 65 ADC  counts across channels
            but the noise RMS stays constant at -1 count     
      """
      def __init__(self, pedestal_value=50.0, noise_sigma=1.0):
            """
            Args:
                  pedestal value: Mean pedestal in ADC Counts (typical: 35-65)
                  noise_sigma: Electronic noise RMS in ADC counts (~1.0)
            """
            self.pedestal_value = pedestal_value
            self.noise_sigma = noise_sigma

      def get_pedestal_for_signal_event(self, samples):
            """
            For signal events, pedestal is the average of first and last samples.
            This is valid because the signla time domain is smaller that the full sampling
            window (7x25=125ns), so the first and last samples contain no signal
            """
            return 0.5*(samples[0] + samples[1])
      
      def get_pedestal_for_pedestal_event(self, samples):
            """
            for pedestal only events (no signal), use average of all samples.
            """
            return np.mean(samples)
      
      def subtract_pedestal(self, samples, is_signal=True):
            """Subtract the pedestal from samples."""
            if is_signal:
                  ped = self.get_pedestal_for_signal_event(samples)
            else:
                  ped = self.get_pedestal_for_pedestal_event(samples)
            return samples-ped, ped

#### **NOISE AUTOCORRELATION MATRIX "R"**

Three options for R:

A) Analytical (thermal noise only)

B) From pedestal data using 

$R_{ij} =\frac{\sum (n_i-\langle n_i\rangle)(n_j-\langle n_j\rangle)}{\sqrt{\sum (n_i-\langle n_i\rangle)^2 \; \sum (n_j-\langle n_j\rangle)^2}}=\frac{\left\langle (n_i-\langle n_i\rangle)(n_j-\langle n_j\rangle)\right\rangle}
{\sqrt{\mathrm{Var}(n_i)\,\mathrm{Var}(n_j)}}$

C) Identity matrix (no correlation assumed)

The paper uses option C) ($R=I$) because TileCal electronics show weak sample-to-sample noise correlation.


In [154]:
class NoiseModel:
      """
      Implements noise autocorrelation matrix R.

      With R = I(Identity), the OF weights become proportional to g and g'
      modulated only by the constraints.
      """
      def __init__(self, n_samples=7, noise_sigma=1.0, correlation_type='identity'):
            """
            Args:
                  n_samples: Number of digitized samples (typically 7 or 9)
                  noise_sigma: Electronic noise RMS in ADC counts
                  correlation_type: 'identity' (paper default), 'thermal', or 'from_data'
            """
            self.n_samples = n_samples
            self.noise_sigma = noise_sigma
            self.correlation_type = correlation_type

      def get_R_matrix(self, pedestal_data=None):
            """
            Compute the noise autocorrelation matrix R.

            Type C (identity): R_ij = delta_ij
                  'Assuming there is no correlation thenn R = I'
            
            Type B (from data): Use Eq. (30) on pedestal data
            R_ij = <n_i * n_j> / sqrt(Var(n_i) * Var(n_j))
            """
            if self.correlation_type == 'identity':
                  return np.eye(self.n_samples)
            elif self.correlation_type == 'thermal':
                  # Option A: Approximate thermal noise with mild correlations
                  R = np.eye(self.n_samples)
                  for i in range(self.n_samples):
                        for j in range(self.n_samples):
                              if i != j:
                                    R[i, j] = 0.05 * np.exp(-abs(i - j) / 2.0)
                  return R
            
            elif self.correlation_type == 'from_data' and pedestal_data is not None:
                  # Option B: Compute from pedestal data using Eq. (30)
                  n_events = pedestal_data.shape[0]
                  means = np.mean(pedestal_data, axis=0)
                  centered = pedestal_data - means
                  
                  R = np.zeros((self.n_samples, self.n_samples))
                  variances = np.var(centered, axis=0)
                  
                  for i in range(self.n_samples):
                        for j in range(self.n_samples):
                              cov_ij = np.mean(centered[:, i] * centered[:, j])
                              denom = np.sqrt(variances[i] * variances[j])
                              R[i, j] = cov_ij / denom if denom > 0 else (1.0 if i == j else 0.0)
                  return R
            
            return np.eye(self.n_samples)
      
      def generate_noise(self, n_events=1):
            """Generate noise samples with the given correlation structure."""
            R = self.get_R_matrix()
            # Scale correlation matrix by noise variance
            cov = self.noise_sigma**2 * R
            return np.random.multivariate_normal(
                  np.zeros(self.n_samples), cov, size=n_events
            )

#### **Optimal Filter Weight Calculation**

The core algorithm. We solve two $(n+2) x (n+2)$ linear systems:

For amplitude weights 'a' [Eq. 32]:

$\begin{bmatrix}R & g & g' \\g^{T} & 0 & 0 \\(g')^{T} & 0 & 0\end{bmatrix}\begin{bmatrix}a \\ \lambda \\ \kappa \end{bmatrix} =\begin{bmatrix}0 \\ 1 \\ 0 \end{bmatrix}$

 For time weights 'b' [Eq. 34]:

$\begin{bmatrix}R & g & g' \\ g^{T} & 0 & 0 \\ (g')^{T} & 0 & 0 \end{bmatrix} \begin{bmatrix}b \\ \mu \\ \rho \end{bmatrix} = \begin{bmatrix}0 \\ 0 \\ -1 \end{bmatrix}$

 Constraints [Eq. 13]:

$
\sum_{i=1}^{n} a_i g_i = 1,
\qquad
\sum_{i=1}^{n} b_i g_i = 0
$

$
\sum_{i=1}^{n} a_i g_i' = 0,
\qquad
\sum_{i=1}^{n} b_i g_i' = -1
$

In [155]:
class OptimalFilterWeights:
      """
      Computes Optimal Filter weights by solving the matrix equations.

      The weights depend on: 
            - g: normalized pulse shape sampled at digitization points
            - g': derivative of pulse shape at same points
            - R: noide autocorrelation matrix

      For each reference time, we build and solve a 9x9 system (7 samples + 2 lagrange multipliers)
      to get the amplitude weights 'a' and time weights 'b'.
      'We calculate all the weights of 25 reference times between -12ns to 12ns in steps of 1ns'
      """
      def __init__(self, sff, noise_model, n_samples=7, sampling_period=25.0):
            """
            Args:
                  sff: ShapeFormFunction instance
                  noise_model = NoideModel instance
                  n_samples: Number of digitized samples (7 for tileCal)
                  sampling_period: Time between samples in ns (25 ns for TileCal)
            """
            self.sff = sff
            self.noise_model = noise_model
            self.n_samples = n_samples
            self.sampling_period = sampling_period
            self.R = noise_model.get_R_matrix()

            # Storage for 25 sets of weights
            # Reference rimes from -12 to +12 ns in steps of 1ns
            self.ref_times = np.arange(-12, 13, 1.0) # 25 values
            self.weights_a = {} # amplitude weights indexed by ref_time
            self.weights_b = {} # time weights indexed by ref_time

      def _sample_times(self, ref_time=0.0):
            """
            Calculate the sampling times relative to the pulse peak.
            
            The central sample (index n//2) is placed at the reference time.
            Other samples are spaced by the sampling period (25 ns).
            
            In the paper it is written that "The time distance of two consecutive elements of g
            must be the sampling period" and "the position of the g elements
            in the shape form should be as close as possible as the samples
            position"
            """
            center = self.n_samples // 2  # Central sample index (3 for 7 samples)
            times = np.array([(i - center) * self.sampling_period + ref_time 
                              for i in range(self.n_samples)])
            return times
      
      def _compute_g_vectors(self, ref_time=0.0):
            """
            Compute g and g' vectors at the sampling times.
            
            g_i = SF_g(t_i) from Eq. (38): normalized shape form at sample time i
            g'_i = d(SF_g)/dt at t_i: derivative of normalized shape form
            """
            times = self._sample_times(ref_time)
            g = self.sff.sf_normalized(times)
            gp = self.sff.sf_derivative(times)
            return g, gp
      
      def compute_weights(self, ref_time=0.0):
        """
        Solve the (n+2) x (n+2) matrix system for weights at a given reference time.
        
        For amplitude weights 'a' - Eq. (32)
        For time weights 'b' - Eq. (34):
        
        Returns:
            a: amplitude weights (n values)
            b: time weights (n values)
        """
        n = self.n_samples
        g, gp = self._compute_g_vectors(ref_time)
        R = self.R
        
        # Build the (n+2) x (n+2) matrix [Eq. 32]
        M = np.zeros((n + 2, n + 2))
        
        # Top-left: R matrix (n x n)
        M[:n, :n] = R
        
        # Top-right: g and g' columns
        M[:n, n] = g       # Column n: g vector
        M[:n, n+1] = gp    # Column n+1: g' vector
        
        # Bottom-left: g^T and g'^T rows
        M[n, :n] = g       # Row n: g^T
        M[n+1, :n] = gp    # Row n+1: g'^T
        
        
        # Right-hand side for amplitude weights 'a'
        rhs_a = np.zeros(n + 2)
        rhs_a[n] = 1.0  
        rhs_b = np.zeros(n + 2)
        rhs_b[n+1] = -1.0  
        
        # Solve both systems
        sol_a = solve(M, rhs_a)
        sol_b = solve(M, rhs_b)
        
        # Extract weights (first n components, last 2 are Lagrange multipliers)
        a = sol_a[:n]
        b = sol_b[:n]
        
        return a, b
    
      def compute_all_weights(self):
            """
            Compute weights for all 25 reference times.
            
            "We calculate all the weights of 25 reference times between 
            -12 ns to 12 ns in steps of 1 ns"
            """
            for ref_time in self.ref_times:
                  a, b = self.compute_weights(ref_time)
                  self.weights_a[ref_time] = a
                  self.weights_b[ref_time] = b
            
            print(f"[Computed] OF weights for {len(self.ref_times)} reference times "
                  f"({self.ref_times[0]} to {self.ref_times[-1]} ns)")
    
      def get_weights(self, ref_time=0.0):
            """Get the closest available weight set for a given reference time."""
            closest = min(self.weights_a.keys(), key=lambda x: abs(x - ref_time))
            return self.weights_a[closest], self.weights_b[closest], closest
    
      def verify_constraints(self, ref_time=0.0):
            """
            Verify the four constraints 
            """
            a, b = self.compute_weights(ref_time)
            g, gp = self._compute_g_vectors(ref_time)
            
            c1 = np.dot(a, g)      # Should be 1
            c2 = np.dot(a, gp)     # Should be 0
            c3 = np.dot(b, g)      # Should be 0
            c4 = np.dot(b, gp)     # Should be -1
            
            print(f"\n[Verify] OF constraints at ref_time = {ref_time} ns:")
            print(f"  Σ a_i g_i  = {c1:.6f}  (should be  1.0)")
            print(f"  Σ a_i g'_i = {c2:.6f}  (should be  0.0)")
            print(f"  Σ b_i g_i  = {c3:.6f}  (should be  0.0)")
            print(f"  Σ b_i g'_i = {c4:.6f}  (should be -1.0)")
            return c1, c2, c3, c4

            

def plot_weights(of_weights):
    """Visualize the OF weights vs reference time (reproduces Fig. 5)."""
    ref_times = sorted(of_weights.weights_a.keys())
    n = of_weights.n_samples
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Top-left: All amplitude weights vs reference time
    for i in range(n):
        vals = [of_weights.weights_a[t][i] for t in ref_times]
        axes[0, 0].plot(ref_times, vals, 'o-', markersize=3, label=f'a{i+1}')
    axes[0, 0].set_xlabel('Reference time (ns)')
    axes[0, 0].set_ylabel('Weight value')
    axes[0, 0].set_title('Amplitude weights a_i vs reference time\n[Paper Fig. 3/5]')
    axes[0, 0].legend(fontsize=8, ncol=2)
    
    # Top-right: All time weights vs reference time
    for i in range(n):
        vals = [of_weights.weights_b[t][i] for t in ref_times]
        axes[0, 1].plot(ref_times, vals, 'o-', markersize=3, label=f'b{i+1}')
    axes[0, 1].set_xlabel('Reference time (ns)')
    axes[0, 1].set_ylabel('Weight value')
    axes[0, 1].set_title('Time weights b_i vs reference time\n[Paper Fig. 4/5]')
    axes[0, 1].legend(fontsize=8, ncol=2)
    
    # Bottom-left: Central amplitude weight (a4) like Fig. 5 left
    center = n // 2
    vals_center = [of_weights.weights_a[t][center] for t in ref_times]
    axes[1, 0].plot(ref_times, vals_center, 'bs-', markersize=6)
    axes[1, 0].set_xlabel('Reference time (ns)')
    axes[1, 0].set_ylabel(f'a{center+1} weight')
    axes[1, 0].set_title(f'Central amplitude weight a{center+1} vs ref time\n[Paper Fig. 5 left]')
    
    # Bottom-right: Central time weight (b4) like Fig. 5 right
    vals_center_b = [of_weights.weights_b[t][center] for t in ref_times]
    axes[1, 1].plot(ref_times, vals_center_b, 'rs-', markersize=6)
    axes[1, 1].set_xlabel('Reference time (ns)')
    axes[1, 1].set_ylabel(f'b{center+1} weight')
    axes[1, 1].set_title(f'Central time weight b{center+1} vs ref time\n[Paper Fig. 5 right]')
    
    plt.tight_layout()
    plt.savefig('fig_of_weights.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("[Fig saved] OF weights vs reference time (Section 3.4)")
   

#### **SIGNAL RECONSTRUCTION WITH OF**

The reconstructed amplitude and time are:

$ \ \ \ u = \sum{a_i \ S_i}$     ->  amplitude estimate 

$ \ \ \ u = \sum{b_i \ S_i}$     ->  time estimate

$ \ \ \ \tau = \frac{v}{u}$     ->  reconstructed phase

The Optimal Filter minimizes Var(u) and Var(v) subject to:

$\langle u \rangle = A$ and $ \langle v \rangle = \tau$

In [156]:
class OptimalFilterReconstructor:
      """
      Applies the OF algorithm to reconstruct amplitude and time from samples.
            'OF Consists in a weighted sum of the signal samples to recover its 
            its parameters (samplitude and time) while minimizing the noise impact.'
      """
      def __init__(self, of_weights, pedestal_calc):
            """
            Args:
                  of_weights: OptimalFilterWeights instance with computed weights
                  pedestal_calc: PedestalCalclator instance
            """
            self.of_weights = of_weights
            self.pedestal_calc = pedestal_calc

      def reconstruct_simple(self, samples, ref_time=0.0, subtract_ped=True):
            """
            Simple (single-pass) OF Reconstruction.
                  - u = sum(a_i * S_i)               (amplitude)
                  - v = sum(b_i * S_i)               (related to time)
            Used for CIS events where the phase is known.

            Args:
                  samples: Array of n digitized sample values
                  ref_time: Reference time for weight selection
                  subtract_ped: whether to subtract pedestal first

            Returns:
                  amplitude: Reconstructed amplitude (energy proxy)
                  phase: Reconstructed phase in ns
                  pedestal: Subtracted pedestal value 
            """
            if subtract_ped:
                  samples_sub, ped = self.pedestal_calc.subtract_pedestal(samples, 
                                                                          is_signal=True)
            else:
                  samples_sub = samples
                  ped = 0.0
            
            a, b, actual_ref = self.of_weights.get_weights(ref_time)
        
            # amplitude reconstruction
            amplitude = np.dot(a, samples_sub)
            # time reconstruction
            v = np.dot(b, samples_sub)
            
            # Phase = v / amplitude 
            phase = v / amplitude if abs(amplitude) > 1e-10 else 0.0
            
            return amplitude, phase, ped
      
      def reconstruct_iterative(self, samples, subtract_ped=True, max_iter=10):
            """
            Iterative OF Reconstruction for Physisc events

            The iteration procedure:
                  1. Check if signal exists (pedestal detection)
                  2. Shift samples if maximum not centered
                  3. Apply OF with ref_time = 0
                  4. If tau < 0.5ns, stop. Otherwise pick closest weights and repeat.

            'The solution to the problem is to apply the prpoer weights for each event 
            according to eht position of the samples in the SFF.'

            Args:
                  samples: Raw digitized samples
                  subtract_ped: whether to subtract pedestal
                  max_iter: Maximum iterations allowed

            Returns: 
                  amplitudes: Reconstructed amplitude
                  total phase: Reconstructed absolute phase (ns)
                  pedestal: Subtracted pedestal value
                  n_iterations: Number of iterations performed
                  is_pedestal: whether event was flagged as pedestal
            """
            n = len(samples)
            center = n //2

            # step 0: pedestal detection, algorithms flags an event as pedestal when neither the 
            # maximum sample is the central sampled nor the two samples next to it
            max_idx = np.argmax(samples)
            max_val = samples[max_idx]

            # Check if max is near center
            is_near_center = abs(max_idx - center) <= 1

            # Also check if its a pedestal event: 'difference of the maximum samples of the 
            # first or last samples is smaller than 4 counts'
            # with noise sigma ~1.5, use threshold og ~5*sigma to be safe
            min_val = np.min(samples)
            is_pedestal = (max_val - min_val) < 8.0

            if is_pedestal:
                  if subtract_ped:
                        _, ped = self.pedestal_calc.subtract_pedestal(samples, is_signal=False)
                  else:
                        ped = 0.0
                  return 0.0, 0.0, ped, 0, True

            # step-1: shift samples if needed 
            # "If the maximum sample is not places in the cecntral 
            # sample the samples are shifted a proper number fo positions"
            n_shifts = max_idx - center
            shifted_samples = np.roll(samples, -n_shifts)

            # subtract pedestal 
            if subtract_ped:
                  samples_sub, ped = self.pedestal_calc.subtract_pedestal(
                        shifted_samples, is_signal=True
                  )
            else:
                  samples_sub = shifted_samples.copy()
                  ped = 0.0

            # step-2: First iteration with refrence time 0
            current_ref = 0.0
            amplitude = 0.0
            phase = 0.0

            for iteration in range(max_iter):
                  a, b, actual_ref = self.of_weights.get_weights(current_ref)

                  amplitude = np.dot(a, samples_sub)

                  v = np.dot(b, samples_sub)
                  phase = v / amplitude if abs(amplitude) > 1e-10 else 0.0

                  # step-3: check convergence 
                  # 'If the reconstructed phase in this iteration is within 
                  # -0.5ns and 0.5ns the process stops'
                  if abs(phase) <= 0.5:
                        break

                  # step-4 update the reference time 
                  # 'the process is repeated with weights of the closest reference
                  # time to the reconstructed phase'
                  new_ref = actual_ref + phase
                  new_ref = np.clip(new_ref, -12, 12)
                  current_ref = round(new_ref)

            # converting to absolute phase
            # 'the number of shifts times 25ns should be added to the total
            # reconstructed phase'
            total_phase = phase + actual_ref + n_shifts * self.of_weights.sampling_period
            return amplitude, total_phase, ped, iteration + 1, False  

#### **Flat Filtering (FF) - Comparision Algorithm**

FF is the Tilecal test beam algorithm. It sums the 5 consecutive samples that maximize the signal.

$E \ = \ max[\sum_{i=j}^{j+4}(S_i-S_1)] \ \ \ \ j = 1, . . . , 5.$

*The FF algorithm consisits of a plain sum of the samples once the pedestal defined as the signal at the first sample, has been subtracted*


In [157]:
class FlatFilterReconstructor:
      """
      Flat Filtering (FF) algorithm.
      The simplest reconstruction: sliding window of 5 samples.
      No pulse shape knowledge, no noise optimization.
      Output is proportional to the area under the pulse.
      """

      def __init__(self):
            pass

      def reconstruct(self, samples):
            """
            Pedestal = first sample value.
            
            Returns:
            amplitude: Reconstructed energy (area-based)
            pedestal: The pedestal (first sample)
            """
            # Pedestal is the first sample
            pedestal = samples[0]
            samples_sub = samples - pedestal
            
            n = len(samples_sub)
            window_size = 5
            max_sum = -np.inf
            
            # Slide a window of 5 and pick the maximum sum
            for j in range(n - window_size + 1):
                  window_sum = np.sum(samples_sub[j:j + window_size])
                  if window_sum > max_sum:
                        max_sum = window_sum
            
            return max_sum, pedestal

#### **Fit Method (FM) - Comparision Algorithm**

Fit method reconstructs energy and tie by fitting the samples to the known pulse shape, minimizing chi-square.

Equivalent to the first order with OF technique

In [158]:
class FitMethodReconstructor:
    """
    Fit Method (FM) algorithm.

    Fits the pulse shape function to the measured samples by χ² minimization.
    Simultaneously extracts amplitude, time, and pedestal.
    More compute-intensive than OF but serves as a benchmark.
    """

    def __init__(self, sff, n_samples=7, sampling_period=25.0):
        self.sff = sff
        self.n_samples = n_samples
        self.sampling_period = sampling_period
        self.center = n_samples // 2

    def _model(self, sample_indices, amplitude, t_offset, pedestal):
        """Model prediction: ped + A * g(t_i + t_offset)"""
        times = (sample_indices - self.center) * self.sampling_period + t_offset
        return pedestal + amplitude * self.sff.sf_normalized(times)

    def reconstruct(self, samples, noise_sigma=1.0):
        """
        Fit the pulse shape to the samples by minimizing χ².
        
        Returns:
            amplitude: Fitted amplitude
            phase: Fitted time offset (ns)
            pedestal: Fitted pedestal
        """
        indices = np.arange(self.n_samples, dtype=float)
        
        # Initial guesses
        ped_guess = 0.5 * (samples[0] + samples[-1])
        amp_guess = np.max(samples) - ped_guess
        t_guess = 0.0
        
        try:
            popt, _ = curve_fit(
                self._model, indices, samples,
                p0=[amp_guess, t_guess, ped_guess],
                bounds=([0, -25, -200], [1e6, 25, 300]),
                maxfev=5000
            )
            return popt[0], popt[1], popt[2]
        except:
            # If fit fails, return pedestal-subtracted max as amplitude
            return max(amp_guess, 0), 0.0, ped_guess

#### **Data Simulation Engine**

Since we don't have actual TileCal data files, we simulate realistic data matching the conditions described in the paper:

- CIS events with known injected charge and variable phase
- Physics events (pions, electrons, muons) with random phase
- Pedestal events (noise only)

In [159]:
class TileCalSimulator:
      """
      Simulates realistic TileCal digitized data.

      CIS events:
            - Injected charge from : Q = 2C * 4.096 * N_DAC/1023
            - Phase sweeps from -12 to +12 ns
            - 5 pF cap: 0-40pC, 100 pF cap: 0-800 pC

      Physics events:
            - Pions, electrons: 10-400 GeV
            - Muons: minimum ionizing (~1-2 GeV equivalent in a cell)
            - Asynchronous: random phase (~12 to +12 ns uniform)
      """

      def __init__(self, sff, pedestal_value=50.0, noise_sigma=1.5, n_samples=7,
                   sampling_period=25.0):
            self.sff = sff
            self.pedestal_value = pedestal_value
            self.noise_sigma = noise_sigma
            self.n_samples = n_samples
            self.sampling_period = sampling_period
            self.center = n_samples // 2

            # CIS calibration constant: ADC counts per pC
            # Typical TileCal value ~1.0 pC/ADC count for high gain
            self.counts_per_pC_hg = 81.0 # High gain: ~81 counts/pC
            self.counts_per_pC_lg = 1.6 # Low gain : ~1.6 counts/pC

      def generate_cis_events(self, charges_pC, phases_ns=None, gain='high'):
            """
            Generate CIS calibration events

            The CIS system discharges a capacitor at configurable phases from 
            -12 to +12 ns relative to the digiter clock.

            Args:
                  charges_pC: Array of injected charges in picocouloumbs
                  phase_ns: Array of phases in ns (default: sweep -12 to +12)
                  gain: 'high' or 'low'

            Returns:
                  all_samples: Array of shape (n_events, n_samples)
                  true_amplitudes: True amplitude in ADC counts
                  true_phases: True phases in ns
            """
            if phases_ns is None:
                  # Sweep phases like CIS does (use 1 ns steps for speed)
                  phases_ns = np.arange(-12, 12.1, 1.0)

            counts_per_pC = self.counts_per_pC_hg if gain == 'high' else self.counts_per_pC_lg

            all_samples = []
            true_amplitudes = []
            true_phases = []

            for Q in charges_pC:
                  for phase in phases_ns:
                        # True amplitude in ADC counts 
                        A_true = Q * counts_per_pC

                        # Sample times relative to pulse peak
                        times = np.array([(i - self.center) * self.sampling_period - phase for i in range(self.n_samples)])

                        # Build sample: pedestal + amplitude * g(t) + noise
                        g_vals = self.sff.sf_normalized(times)
                        noise = np.random.normal(0, self.noise_sigma, self.n_samples)
                        samples = self.pedestal_value + A_true * g_vals + noise
                        all_samples.append(samples)
                        true_amplitudes.append(A_true)
                        true_phases.append(phase)

            return (np.array(all_samples), np.array(true_amplitudes), np.array(true_phases))

      def generate_physics_events(self, energies_GeV, n_events_per_energy=100, particle='pion'):
            """
            Generate physics events

            Physics events have:
                  - Random phase (asynchronous beam): uniform in [-12, +12] ns
                  - Energy deposited propotional to beam energy
                  - Shape from ~10% wider than CIS (accounted for by sff choice)

            Args:
                  energies_GeV: Array of beam energies
                  n_events_per_energy: Number of events per energy point
                  particle: 'pion', 'electron' or 'muon'

            Returns:
                  all_samples, true_amplitudes, true_phase, true_engines  
            """         
            # approximate energy to ADC count conversion
            # These are rough but realistic for TileCal cells
            if particle == 'muon':
                  # Muons are MPI: deposit ~1-2GeV equivalent per cell
                  adc_per_GeV = 25.0 # Low signal
            elif particle == 'electron':
                  # Electrons shower electromagnetically
                  adc_per_GeV = 50.0
            else:
                  # Pions: hadronic shower
                  adc_per_GeV = 35.0
            all_samples = []
            true_amplitudes = []
            true_phases = []
            true_energies = []

            for E in energies_GeV:
                  for _ in range(n_events_per_energy):
                        # True amplitude
                        A_true = E * adc_per_GeV

                        # Random phase (asynchronous beam)
                        phase = np.random.uniform(-12, 12)

                        # Samples times
                        times = np.array([(i - self.center) * self.sampling_period - phase for i in range(self.n_samples)])
                        g_vals = self.sff.sf_normalized(times)
                        noise = np.random.normal(0, self.noise_sigma, self.n_samples)
                        samples = self.pedestal_value + A_true * g_vals + noise

                        all_samples.append(samples)
                        true_amplitudes.append(A_true)
                        true_phases.append(phase)
                        true_energies.append(E)

            return (np.array(all_samples), np.array(true_amplitudes),
                    np.array(true_phases), np.array(true_energies))
      
      def generate_pedestal_events(self, n_events=1000):
            """
            Generate pedestal-only events (noise, no signal).
            Used for noise studies and R matrix calculation
            """
            noise = np.random.normal(0, self.noise_sigma, (n_events, self.n_samples))
            samples = self.pedestal_value + noise
            return samples

#### **CIS Calibration Analysis**

In [160]:
def run_cis_calibration(sff, of_weights, simulator, ped_calc):
      """
      CIS caibration analysis

      'For each charge we average the output in ADC counts of the algorithms
      for all the phases and make linear fit'

      Computer calibration factors: counts/pC for both OF and FF methods.
      """
      # CIS charges using 100 pF capacitor
      # Q = 2 * C * 4.096 * N_DAC / 1023
      # For high gain: 0 to 12 pC range
      charges_hg = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0])

      print(f"  Charges (high gain): {charges_hg.min():.1f} to {charges_hg.max():.1f} pC")

      # Generates CIS events with phase sweep
      samples_all, true_amps, true_phases = simulator.generate_cis_events(
            charges_hg, gain='high')
      
      # Number of phases per charge
      n_phases = len(np.arange(-12, 12.1, 1.0))

      # Reconstruct with OF and FF
      of_recon = OptimalFilterReconstructor(of_weights, ped_calc)
      ff_recon = FlatFilterReconstructor()

      of_means = []
      ff_means = []
      charge_list = []

      for i, Q in enumerate(charges_hg):
            idx_start = i * n_phases
            idx_end = (i+1) * n_phases

            of_amps = []
            ff_amps = []

            for j in range(idx_start, min(idx_end, len(samples_all))):
                  s = samples_all[j]
                  phase_true = true_phases[j]

                  # OF reconstruction (CIS: known phase, use closest of ref time)
                  # ref_time = -phase_true beacuse of sign convention:
                  # phase > 0 means signal arrived laterm so peak is at -phase in weight space
                  amp_of, _, _ = of_recon.reconstruct_simple(s, ref_time=round(-phase_true))
                  of_amps.append(amp_of)
                  
                  # FF reconstruction
                  amp_ff, _ = ff_recon.reconstruct(s)
                  ff_amps.append(amp_ff)

            if len(of_amps) > 0:
                  of_means.append(np.mean(of_amps))
                  ff_means.append(np.mean(ff_amps))
                  charge_list.append(Q)

      charge_list = np.array(charge_list)
      of_means = np.array(of_means)
      ff_means = np.array(ff_means)

      # Linear fit to get calibration constants
      # make a liner fit of the output vs injected charge
      of_fit = np.polyfit(charge_list, of_means, 1)
      ff_fit = np.polyfit(charge_list, ff_means, 1)

      print(f"\n  OF calibration: {of_fit[0]:.2f} counts/pC (intercept: {of_fit[1]:.2f})")
      print(f"  FF calibration: {ff_fit[0]:.2f} counts/pC (intercept: {ff_fit[1]:.2f})")

      fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
      # PLot CIS calibration
      axes[0].plot(charge_list, of_means, 'bo', markersize=4, label='OF data')
      axes[0].plot(charge_list, np.polyval(of_fit, charge_list), 'b-', label=f'OF fit: {of_fit[0]:.1f} cts/pC')
      axes[0].plot(charge_list, ff_means, 'r^', markersize=4, label='FF data')
      axes[0].plot(charge_list, np.polyval(ff_fit, charge_list), 'r--', label=f'FF fit: {ff_fit[0]:.1f} cts/pC')
      axes[0].set_xlabel('Injected charge (pC)')
      axes[0].set_ylabel('Reconstructed amplitude (ADC counts)')
      axes[0].set_title('CIS Calibration: Output vs Charge\n[Paper Fig. 8 / Table 1]')
      axes[0].legend()

      # Residuals
      of_residual = (of_means - np.polyval(of_fit, charge_list)) / np.polyval(of_fit, charge_list) * 100
      ff_residual = (ff_means - np.polyval(ff_fit, charge_list)) / np.polyval(ff_fit, charge_list) * 100
      
      axes[1].plot(charge_list, of_residual, 'bo-', markersize=4, label='OF')
      axes[1].plot(charge_list, ff_residual, 'r^-', markersize=4, label='FF')
      axes[1].set_xlabel('Injected charge (pC)')
      axes[1].set_ylabel('Residual (%)')
      axes[1].set_title('CIS Calibration Residuals\n[Paper Fig. 9 middle]')
      axes[1].legend()
      axes[1].axhline(y=0, color='gray', linestyle='--')
      
      plt.tight_layout()
      plt.savefig('fig_cis_calibration.png', dpi=150, bbox_inches='tight')
      plt.close()
      print("[Fig saved] CIS calibration (Section 5.1)")

      return of_fit, ff_fit

#### **Physics Performace Analysis**

In [161]:
def run_physics_analysis(sff, of_weights, simulator, ped_calc, particle='pion'):
      """
      Physics even analysis

      Compares OF1 OF2, FF, and FM algorithms on physics events.
      Evaluates energy ereolution as a function of beam energy.

      'results are promising specially in the regions where the electronic
      noise contributes significantly to the resolution'
      """
      # Energy points matching the paper
      if particle == 'pion':
            energies = np.array([20, 50, 100, 150, 180, 250, 350])
      elif particle == 'electron':
            energies = np.array([10, 20, 50, 100, 180])
      else:  # muon
            energies = np.array([180])  # Single muon run
      
      n_per_energy = 100

      # Generate events
      samples_all, true_amps, true_phases, true_energies = simulator.generate_physics_events(energies, n_per_energy, particle)

      # Reconstructors
      of_recon = OptimalFilterReconstructor(of_weights, ped_calc)
      ff_recon = FlatFilterReconstructor()
      fm_recon = FitMethodReconstructor(sff)

      # Results storage
      results = {alg: {'energies': [], 'reco_amps': [], 'true_amps': []} 
                  for alg in ['OF1', 'FF', 'FM']}
      
      print(f"  Processing {len(samples_all)} events...")

      for idx in range(len(samples_all)):
            s = samples_all[idx]
            A_true = true_amps[idx]
            E_true = true_energies[idx]
            
            # OF1: Iterative optimal filtering
            amp_of, phase_of, ped_of, n_iter, is_ped = \
                  of_recon.reconstruct_iterative(s)
            results['OF1']['reco_amps'].append(amp_of)
            results['OF1']['true_amps'].append(A_true)
            results['OF1']['energies'].append(E_true)
            
            # FF: Flat filtering
            amp_ff, _ = ff_recon.reconstruct(s)
            results['FF']['reco_amps'].append(amp_ff)
            results['FF']['true_amps'].append(A_true)
            results['FF']['energies'].append(E_true)
            
            # FM: Fit method
            amp_fm, _, _ = fm_recon.reconstruct(s)
            results['FM']['reco_amps'].append(amp_fm)
            results['FM']['true_amps'].append(A_true)
            results['FM']['energies'].append(E_true)

      # Convert to arrays
      for alg in results:
            for key in results[alg]:
                  results[alg][key] = np.array(results[alg][key])

      # Compute energy resolution
      # Resolution = sigma(E_reco) / <E_reco> for each energy bin
      resolution = {alg: {'E': [], 'res': [], 'res_err': []} for alg in results}

      for E in energies:
        for alg in results:
            mask = results[alg]['energies'] == E
            reco = results[alg]['reco_amps'][mask]
            true = results[alg]['true_amps'][mask]
            
            if len(reco) > 10:
                # Relative resolution
                ratio = reco / true if alg != 'FF' else reco / np.mean(reco) * np.mean(true/true)
                mean_r = np.mean(reco)
                std_r = np.std(reco)
                
                if abs(mean_r) > 1e-10:
                    res = std_r / abs(mean_r) * 100  # In percent
                    res_err = res / np.sqrt(2 * len(reco))
                    resolution[alg]['E'].append(E)
                    resolution[alg]['res'].append(res)
                    resolution[alg]['res_err'].append(res_err)
    
      # Plot energy resolution
      fig, axes = plt.subplots(1, 2, figsize=(14, 6))
      
      colors = {'OF1': 'blue', 'FF': 'red', 'FM': 'green'}
      markers = {'OF1': 'o', 'FF': 's', 'FM': '^'}
      
      for alg in resolution:
            E = resolution[alg]['E']
            res = resolution[alg]['res']
            if len(E) > 0:
                  axes[0].errorbar(E, res, yerr=resolution[alg]['res_err'],
                              fmt=f'{markers[alg]}-', color=colors[alg],
                              label=alg, markersize=6, capsize=3)
      
      axes[0].set_xlabel('Beam energy (GeV)')
      axes[0].set_ylabel('Energy resolution σ/E (%)')
      axes[0].set_title(f'Energy Resolution: {particle.capitalize()}s')
      axes[0].legend()
      axes[0].set_yscale('log')
      
      # Energy distribution at one energy point 
      E_show = energies[len(energies)//2]
      for alg in results:
            mask = results[alg]['energies'] == E_show
            reco = results[alg]['reco_amps'][mask]
            if len(reco) > 0:
                  axes[1].hist(reco, bins=30, alpha=0.5, label=alg, color=colors[alg])
      
      axes[1].set_xlabel('Reconstructed amplitude (ADC counts)')
      axes[1].set_ylabel('Events')
      axes[1].set_title(f'Energy Distribution at {E_show} GeV')
      axes[1].legend()
      
      plt.tight_layout()
      plt.savefig(f'fig_physics_{particle}.png', dpi=150, bbox_inches='tight')
      plt.close()
      print(f"[Fig saved] Physics analysis for {particle}s")
      
      return results, resolution

#### **Noise Anallysis**

In [162]:
def run_noise_analysis(sff, of_weights, simulator, ped_calc):
      """
      Noise performance analysis.
      
      "Energy reconstruction in pC in a cell with no deposited energy 
      obtained with the four algorithms"
      
      Compares the noise floor of each algorithm using pedestal events.
      """
      
      # Generate pedestal events (no signal)
      ped_samples = simulator.generate_pedestal_events(n_events=2000)
      
      of_recon = OptimalFilterReconstructor(of_weights, ped_calc)
      ff_recon = FlatFilterReconstructor()
      fm_recon = FitMethodReconstructor(sff)
      
      noise_of = []
      noise_ff = []
      noise_fm = []
      
      for s in ped_samples:
            # OF: Direct application with ref_time=0 (pedestal events have no signal)
            # For noise study, we subtract the mean pedestal and apply weights directly
            # This is how OF noise is characterized
            s_sub = s - np.mean(s)  # Subtract true pedestal
            a, b, _ = of_weights.get_weights(0.0)
            amp_of = np.dot(a, s_sub)
            noise_of.append(amp_of)
            
            # FF
            amp_ff, _ = ff_recon.reconstruct(s)
            noise_ff.append(amp_ff)
            
            # FM
            amp_fm, _, _ = fm_recon.reconstruct(s)
            noise_fm.append(amp_fm)
      
      noise_of = np.array(noise_of)
      noise_ff = np.array(noise_ff)
      noise_fm = np.array(noise_fm)
      
      print(f"Noise RMS (ADC counts):")
      print(f"   OF: {np.std(noise_of):.3f}")
      print(f"   FF: {np.std(noise_ff):.3f}")
      print(f"   FM: {np.std(noise_fm):.3f}")
      print(f"  (OF should show lowest noise")
      
      # Plot noise distributions
      fig, ax = plt.subplots(1, 1, figsize=(10, 6))
      
      bins = np.linspace(-10, 10, 80)
      ax.hist(noise_of, bins=bins, alpha=0.6, label=f'OF (σ={np.std(noise_of):.2f})', color='blue')
      ax.hist(noise_ff, bins=bins, alpha=0.6, label=f'FF (σ={np.std(noise_ff):.2f})', color='red')
      ax.hist(noise_fm, bins=bins, alpha=0.6, label=f'FM (σ={np.std(noise_fm):.2f})', color='green')
      
      ax.set_xlabel('Reconstructed amplitude (ADC counts)')
      ax.set_ylabel('Events')
      ax.set_title('Noise Distribution: No Deposited Energy')
      ax.legend()
      
      plt.tight_layout()
      plt.savefig('fig_noise_analysis.png', dpi=150, bbox_inches='tight')
      plt.close()
      print("[Fig saved] Noise analysis (Section 5.5)")
      
      return noise_of, noise_ff, noise_fm

#### **CIS Phase & Amplitude Reconstruction**

In [163]:
def run_cis_reconstruction_study(sff, of_weights, simulator, ped_calc):
      """
      CIS amplitude and phase reconstruction.
      
      Studies reconstruction performance as a function of injected charge
      for both high and low gain.
      
      "CIS: Charge reconstruction" and "CIS: Phase reconstruction"
      """
      
      # Fixed charge values for the study
      charges_pC = np.array([2, 4, 6, 8, 10, 12])  # High gain range
      
      of_recon = OptimalFilterReconstructor(of_weights, ped_calc)
      ff_recon = FlatFilterReconstructor()
      
      fig, axes = plt.subplots(2, 2, figsize=(14, 10))
      
      for gain_idx, gain in enumerate(['high']):
            samples_all, true_amps, true_phases = simulator.generate_cis_events(
                  charges_pC, gain=gain)
            
            n_phases = len(np.arange(-12, 12.1, 1.0))
            
            # Amplitude reconstruction study
            for q_idx, Q in enumerate(charges_pC):
                  of_amps = []
                  ff_amps = []
                  of_phases_reco = []
                  phases_true = []
                  
                  for j in range(q_idx * n_phases, min((q_idx + 1) * n_phases, len(samples_all))):
                        s = samples_all[j]
                        phase_true = true_phases[j]
                        
                        amp_of, phase_of, _ = of_recon.reconstruct_simple(s, ref_time=round(-phase_true))
                        amp_ff, _ = ff_recon.reconstruct(s)
                        
                        of_amps.append(amp_of)
                        ff_amps.append(amp_ff)
                        of_phases_reco.append(phase_of)
                        phases_true.append(phase_true)
                        
                  of_amps = np.array(of_amps)
                  ff_amps = np.array(ff_amps)
                  
                  if len(of_amps) > 0:
                        # Resolution for this charge
                        A_true = true_amps[q_idx * n_phases]
                        
                        if abs(np.mean(of_amps)) > 0:
                              of_res = np.std(of_amps) / abs(np.mean(of_amps)) * 100
                              ff_res = np.std(ff_amps) / abs(np.mean(ff_amps)) * 100
                              
                              axes[0, 0].plot(Q, of_res, 'bo', markersize=6)
                              axes[0, 0].plot(Q, ff_res, 'rs', markersize=6)
            
            # Plot labels
            axes[0, 0].set_xlabel('Injected charge (pC)')
            axes[0, 0].set_ylabel('Resolution σ/E (%)')
            axes[0, 0].set_title('CIS Charge Resolution')
            axes[0, 0].plot([], [], 'bo', label='OF')
            axes[0, 0].plot([], [], 'rs', label='FF')
            axes[0, 0].legend()
      
      # Phase reconstruction study
      Q_fixed = 6.0  # Fixed charge for phase study
      samples_all, true_amps, true_phases = simulator.generate_cis_events(
            [Q_fixed], gain='high')
      
      of_phase_reco = []
      of_phase_true = []
      
      for j in range(len(samples_all)):
            s = samples_all[j]
            phase_true = true_phases[j]
            _, phase_of, _ = of_recon.reconstruct_simple(s, ref_time=round(-phase_true))
            of_phase_reco.append(phase_of + round(-phase_true))  # Absolute phase
            of_phase_true.append(phase_true)
      
      of_phase_reco = np.array(of_phase_reco)
      of_phase_true = np.array(of_phase_true)
      
      # Phase reconstruction scatter
      axes[0, 1].scatter(of_phase_true, of_phase_reco, s=5, alpha=0.5)
      axes[0, 1].plot([-12, 12], [-12, 12], 'r--', label='Perfect')
      axes[0, 1].set_xlabel('True phase (ns)')
      axes[0, 1].set_ylabel('Reconstructed phase (ns)')
      axes[0, 1].set_title('CIS Phase Reconstruction')
      axes[0, 1].legend()
      
      # Phase residual
      phase_residual = of_phase_reco - of_phase_true
      axes[1, 0].scatter(of_phase_true, phase_residual, s=5, alpha=0.5)
      axes[1, 0].axhline(y=0, color='r', linestyle='--')
      axes[1, 0].set_xlabel('True phase (ns)')
      axes[1, 0].set_ylabel('Phase residual (ns)')
      axes[1, 0].set_title('Phase Residual vs True Phase')
      
      # Phase residual distribution
      axes[1, 1].hist(phase_residual, bins=50, edgecolor='black', alpha=0.7)
      axes[1, 1].set_xlabel('Phase residual (ns)')
      axes[1, 1].set_ylabel('Events')
      axes[1, 1].set_title(f'Phase Residual Distribution\nRMS = {np.std(phase_residual):.3f} ns')
      
      plt.tight_layout()
      plt.savefig('fig_cis_reconstruction.png', dpi=150, bbox_inches='tight')
      plt.close()
      print("[Fig saved] CIS reconstruction study")
      

#### **CIS Quality Factor**

In [164]:
def run_quality_factor_study(sff, of_weights, simulator, ped_calc):
        """
        Quality factor study.
        
        The quality factor QF measures the goodness of the OF reconstruction:
        QF = sum (S_i - A*g_i - ped)^2
        
        It is equivalent to chi-square and measures how well the reconstructed parameters
        describe the actual samples.
        """
        charges_pC = np.array([2, 4, 6, 8, 10, 12])
        of_recon = OptimalFilterReconstructor(of_weights, ped_calc)
        
        samples_all, true_amps, true_phases = simulator.generate_cis_events(
            charges_pC, gain='high')
        
        n_phases = len(np.arange(-12, 12.1, 1.0))
        qf_values = []
        charge_values = []
        
        for q_idx, Q in enumerate(charges_pC):
            for j in range(q_idx * n_phases, min((q_idx + 1) * n_phases, len(samples_all))):
                s = samples_all[j]
                phase_true = true_phases[j]
                
                amp_of, phase_of, ped = of_recon.reconstruct_simple(s, ref_time=round(-phase_true))
                
                # Reconstruct expected samples
                times = of_weights._sample_times(round(-phase_true) + phase_of)
                g_expected = sff.sf_normalized(times)
                s_expected = ped + amp_of * g_expected
                
                # Quality factor = sum (S_i - S_expected_i)^2
                qf = np.sum((s - ped - amp_of * g_expected)**2)
                qf_values.append(qf)
                charge_values.append(Q)
        
        qf_values = np.array(qf_values)
        charge_values = np.array(charge_values)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # QF distribution
        axes[0].hist(qf_values, bins=50, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Quality Factor')
        axes[0].set_ylabel('Events')
        axes[0].set_title('Quality Factor Distribution')
        
        # QF vs charge
        for Q in charges_pC:
            mask = charge_values == Q
            axes[1].scatter([Q] * np.sum(mask), qf_values[mask], s=2, alpha=0.3)
        axes[1].set_xlabel('Injected charge (pC)')
        axes[1].set_ylabel('Quality Factor')
        axes[1].set_title('Quality Factor vs Charge')
        
        plt.tight_layout()
        plt.savefig('fig_quality_factor.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"  Mean QF: {np.mean(qf_values):.2f}")
        print(f"  Median QF: {np.median(qf_values):.2f}")
        print("[Fig saved] Quality factor")

In [165]:
def print_summary_table(of_fit, ff_fit, noise_of, noise_ff, noise_fm):
      """
      Print a summary table (there is one in the paper) (just a small comaparision).
      """
      print(f"\n{'='*70}")
      print("SUMMARY TABLE")
      print("="*70)
      print(f"{'Metric':<40} {'OF':>10} {'FF':>10} {'FM':>10}")
      print("-"*70)
      print(f"{'CIS calibration (counts/pC)':<40} {of_fit[0]:>10.2f} {ff_fit[0]:>10.2f} {'N/A':>10}")
      print(f"{'Noise RMS (ADC counts)':<40} {np.std(noise_of):>10.3f} {np.std(noise_ff):>10.3f} {np.std(noise_fm):>10.3f}")
      print(f"{'Noise improvement vs FF':<40} {np.std(noise_ff)/np.std(noise_of):>10.2f}x {'1.00x':>10} {np.std(noise_ff)/np.std(noise_fm):>10.2f}x")
      print("-"*70)
      print("Note: OF shows best noise performance (lowest RMS)")
      print("      as expected from the optimal noise minimization")

#### **OF2 (Pedestal as output paramter)** [Appendix A study]

In [166]:
def study_of2_pedestal_reconstruction(sff, of_weights, simulator):
      """
      Brief study of the OF2 variant (Appendix A).
      
      OF2 avoids pedestal pre-calculation by introducing pedestal as a 
      third output parameter. This saves computing time at the ROD level
      "available computing time for the online reconstruction
      at the ROD level is only 10 μs".
      
      The OF2 adds a third set of weights 'c' and additional constraints:
            sum (c_i) = 1 (pedestal weights sum to 1)
            sum (c_i g_i) = 0 (pedestal weights reject signal)
            sum (c_i g'_i) = 0 (pedestal weights reject timing)
      """
      
      print("APPENDIX A: OF2 - PEDESTAL AS OUTPUT PARAMETER")
      
      n = of_weights.n_samples
      g, gp = of_weights._compute_g_vectors(0.0)
      R = of_weights.R
      
      # OF2: (n+3) x (n+3) system with pedestal weights
      # Additional constraint: sum(c_i) = 1 (ones vector)
      ones = np.ones(n)
      
      M = np.zeros((n + 3, n + 3))
      M[:n, :n] = R
      M[:n, n] = g
      M[:n, n+1] = gp
      M[:n, n+2] = ones
      M[n, :n] = g
      M[n+1, :n] = gp
      M[n+2, :n] = ones
      
      # For amplitude: RHS = (0,...,0, 1, 0, 0)
      rhs_a = np.zeros(n + 3)
      rhs_a[n] = 1.0
      
      # For pedestal: RHS = (0,...,0, 0, 0, 1)
      rhs_p = np.zeros(n + 3)
      rhs_p[n+2] = 1.0
      
      sol_a = solve(M, rhs_a)
      sol_p = solve(M, rhs_p)
      
      a_of2 = sol_a[:n]
      p_of2 = sol_p[:n]
      
      print(f"  OF2 amplitude weights: {a_of2}")
      print(f"  OF2 pedestal weights:  {p_of2}")
      
      # Verify
      print(f"\n  Verification:")
      print(f"    sum(a_i g_i) = {np.dot(a_of2, g):.6f} (should be 1)")
      print(f"     sum(p_i) = {np.sum(p_of2):.6f} (should be 1)")
      print(f"    sum(p_i g_i) = {np.dot(p_of2, g):.6f} (should be 0)")
      
      # Test on simulated data
      test_samples = simulator.generate_pedestal_events(100)
      ped_reco = np.array([np.dot(p_of2, s) for s in test_samples])
      
      print(f"\n  Pedestal reconstruction test:")
      print(f"    True pedestal: {simulator.pedestal_value:.1f}")
      print(f"    Mean reconstructed: {np.mean(ped_reco):.2f} +/- {np.std(ped_reco):.2f}")

# **Main Runs**

In [167]:
#Executing the complete Optimal Filtering analysis pipeline,
# following the paper structure from Section 2 through Section 5.

print("="*70)
print("OPTIMAL FILTERING IN THE ATLAS HADRONIC TILE CALORIMETER")
print("ATL-TILECAL-2005-001 — Complete Computational Implementation")
print("="*70)

np.random.seed(42)

# 1. SETUP: Initialize all components
# =====================================================================
print("\n[1/8] Initializing components...")
# Shape Form Function (Section 3.3)
# Typical TileCal parameters
sff_cis = ShapeFormFunction(tau=15.0, mu=7.0, pedestal=0.0, gain='high')
sff_physics = ShapeFormFunction(tau=16.5, mu=7.0, pedestal=0.0, gain='high')
# Physics shape is ~10% wider (tau=16.5 vs 15.0)

# Pedestal calculator (Section 3.1)
ped_calc = PedestalCalculator(pedestal_value=50.0, noise_sigma=1.5)

# Noise model (Section 3.2) - using identity matrix as per the paper
noise_model = NoiseModel(n_samples=7, noise_sigma=1.5, correlation_type='identity')

# Data simulator
simulator = TileCalSimulator(sff_cis, pedestal_value=50.0, noise_sigma=1.5)

print("  Shape Form: tau=15.0 ns, mu=7.0 (CIS)")
print("  Shape Form: tau=16.5 ns, mu=7.0 (Physics, 10% wider)")
print("  Pedestal: 50.0 ADC counts, noise σ=1.5 counts")
print("  Noise model: Identity matrix (R = I)")
print("  Samples: 7 x 25 ns = 175 ns window")

# 2. COMPUTE OF WEIGHTS
# =====================================================================
print("\n[2/8] Computing OF weights...")
    
# CIS weights
of_weights_cis = OptimalFilterWeights(sff_cis, noise_model)
of_weights_cis.compute_all_weights()
of_weights_cis.verify_constraints(0.0)

# Physics weights (different shape form)
of_weights_phys = OptimalFilterWeights(sff_physics, noise_model)
of_weights_phys.compute_all_weights()

# 3. VISUALIZE SHAPE FORM AND WEIGHTS
# =====================================================================
print("\n[3/8] Generating plots...")    
plot_shape_form(sff_cis)
plot_weights(of_weights_cis)

# 4. CIS Calibration
# =====================================================================
print("\n[4/8] CIS calibration analysis...")
of_fit, ff_fit = run_cis_calibration(sff_cis, of_weights_cis, simulator, ped_calc)

# 5. CIS RECONSTRUCTION
# =====================================================================
print("\n[5/8] CIS reconstruction studies...")
run_cis_reconstruction_study(sff_cis, of_weights_cis, simulator, ped_calc)
run_quality_factor_study(sff_cis, of_weights_cis, simulator, ped_calc)

# 6. NOISE CAlibration
# =====================================================================
print("\n[6/8] Noise analysis...")
# Use physics weights and physics simulator for noise study
simulator_phys = TileCalSimulator(sff_physics, pedestal_value=50.0, noise_sigma=1.5)
noise_of, noise_ff, noise_fm = run_noise_analysis(sff_physics, of_weights_phys, simulator_phys, ped_calc)


# 7. PHYSICS ANALYSIS
# =====================================================================

print("\n[7/8] Physics event analysis...")

# Pions
results_pi, res_pi = run_physics_analysis(sff_physics, of_weights_phys, simulator_phys, ped_calc, particle='pion')

# Electrons
results_e, res_e = run_physics_analysis(sff_physics, of_weights_phys, simulator_phys, ped_calc, particle='electron')

# 8. OF2 STUDY (Appendix A)
print("\n[8/8] OF2 pedestal study (Appendix A)...")
study_of2_pedestal_reconstruction(sff_physics, of_weights_phys, simulator_phys)

# Summary table
print_summary_table(of_fit, ff_fit, noise_of, noise_ff, noise_fm)

OPTIMAL FILTERING IN THE ATLAS HADRONIC TILE CALORIMETER
ATL-TILECAL-2005-001 — Complete Computational Implementation

[1/8] Initializing components...
  Shape Form: tau=15.0 ns, mu=7.0 (CIS)
  Shape Form: tau=16.5 ns, mu=7.0 (Physics, 10% wider)
  Pedestal: 50.0 ADC counts, noise σ=1.5 counts
  Noise model: Identity matrix (R = I)
  Samples: 7 x 25 ns = 175 ns window

[2/8] Computing OF weights...
[Computed] OF weights for 25 reference times (-12.0 to 12.0 ns)

[Verify] OF constraints at ref_time = 0.0 ns:
  Σ a_i g_i  = 1.000000  (should be  1.0)
  Σ a_i g'_i = -0.000000  (should be  0.0)
  Σ b_i g_i  = 0.000000  (should be  0.0)
  Σ b_i g'_i = -1.000000  (should be -1.0)
[Computed] OF weights for 25 reference times (-12.0 to 12.0 ns)

[3/8] Generating plots...
[Fig saved] Shape Form Function (Section 3.3)
[Fig saved] OF weights vs reference time (Section 3.4)

[4/8] CIS calibration analysis...
  Charges (high gain): 1.0 to 12.0 pC

  OF calibration: 102.38 counts/pC (intercept: -76.